# Employee attrition · EDA and enrichment

### Standalone notebook | `employee_data.csv`

---

This notebook is independent of the telco case. Same discipline, different data, and
one important difference you need to notice early.

**The telco data had a time dimension. This does not.** `status` tells you whether
someone is employed *now*. There is no prediction point, no observation window, no
way to train on the past and test on the future. That single fact changes what you
are allowed to claim, and section 2 deals with it head on.

### Cell tags

| Tag | What you do |
|---|---|
| **READ** | Run it, read the output, answer the question. No typing. |
| **FILL** | Change one clearly marked value. |
| **BUILD** | Open-ended. Write your own code. |

Everyone completes READ and FILL. If a cell errors, skip it and keep moving.

## 0. Setup and loading

Set `DATA_PATH` to wherever your copy of `employee_data.csv` lives. If the file is
not found, the notebook builds a stand-in with the same schema and the same summary
statistics, so every cell below still runs.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)
plt.rcParams["figure.figsize"] = (10, 4.5)

# FILL: point this at your file.
#   Colab + Google Drive : "/content/drive/MyDrive/employee_data.csv"
#   Colab + direct upload: "employee_data.csv"
#   Local / VS Code      : "data/employee_data.csv"
DATA_PATH = "employee_data.csv"


def build_stand_in(n_rows=14249, seed=7):
    """Same schema and summary statistics as employee_data.csv, for when the file
    is not to hand. Delete this once you have the real file."""
    rng = np.random.default_rng(seed)
    dept = rng.choice(
        ["sales","engineering","support","IT","product","marketing","finance",
         "management","information_technology","temp","admin","procurement"],
        n_rows, p=[.275,.182,.149,.081,.060,.057,.051,.042,.015,.013,.012,.063])
    salary = rng.choice(["low","medium","high"], n_rows, p=[.488,.430,.082])

    satisfaction = np.clip(rng.beta(2.6, 1.6, n_rows), .04, 1.0)
    last_eval    = np.clip(rng.normal(.718, .173, n_rows), .316, 1.0)
    tenure       = np.clip(rng.gamma(4.2, .78, n_rows) + 2, 2, 10).round()
    n_projects   = np.clip(rng.normal(3.77, 1.25, n_rows), 1, 7).round().astype(int)
    hrs          = np.clip(rng.normal(200, 51, n_rows), 49, 310).round().astype(int)
    hrs = np.where(dept == "temp", np.clip(rng.normal(100, 25, n_rows), 49, 160).round(), hrs).astype(int)

    # Three routes out, which is what the real data shows
    overworked_unhappy = (hrs > 245) & (satisfaction < .35)
    coasting           = (last_eval < .55) & (n_projects <= 3)
    burnt_out_star     = (last_eval > .85) & (hrs > 255) & (tenure >= 4)
    p_left = np.clip(.14 + .55*overworked_unhappy + .38*coasting + .48*burnt_out_star
                     + .12*(salary == "low") - .06*(salary == "high"), .02, .95)
    status = np.where(rng.random(n_rows) < p_left, "Left", "Employed")

    df = pd.DataFrame({
        "avg_monthly_hrs": hrs, "department": dept,
        "filed_complaint": np.where(rng.random(n_rows) < .144, 1.0, np.nan),
        "last_evaluation": last_eval.round(6), "n_projects": n_projects,
        "recently_promoted": np.where(rng.random(n_rows) < .021, 1.0, np.nan),
        "salary": salary, "satisfaction": satisfaction.round(6),
        "status": status, "tenure": tenure})

    # the same gaps the real extract has
    df.loc[rng.random(n_rows) < .0498, "department"]      = np.nan
    df.loc[rng.random(n_rows) < .1075, "last_evaluation"] = np.nan
    both = rng.random(n_rows) < .0127
    df.loc[both, "satisfaction"] = np.nan
    df.loc[both, "tenure"]       = np.nan
    return df


if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
    print(f"Loaded {DATA_PATH}")
else:
    df_raw = build_stand_in()
    print(f"'{DATA_PATH}' not found. Using the stand-in dataset.")
    print("Everything below runs identically once you point DATA_PATH at the real file.")

print(f"\n{df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

## 1. Frame the problem before you explore it

Your original notebook opened with `read_csv`. Ninety seconds spent here changes
everything that follows.

| Question | Answer for this dataset |
|---|---|
| What decision does this serve? | Who HR approaches with a retention conversation |
| Who acts on the output? | An HR business partner, with finite time |
| What is the label? | `status == "Left"`. But **left when?** The file does not say. |
| What does a false positive cost? | An unnecessary conversation. Cheap, slightly awkward. |
| What does a false negative cost? | Replacement cost: commonly quoted at 6 to 9 months of salary. |

**Write down the asymmetry.** A false negative here costs perhaps fifty times a
false positive. That ratio, not accuracy, is what should drive every threshold
decision later.

In [ ]:
# FILL: your own cost assumptions. Everything downstream depends on these.
COST_FALSE_POSITIVE = 5_000       # KES: one awkward, unnecessary conversation
COST_FALSE_NEGATIVE = 600_000     # KES: recruit, onboard and ramp a replacement

print(f"A missed leaver costs {COST_FALSE_NEGATIVE/COST_FALSE_POSITIVE:.0f}x "
      f"an unnecessary conversation.")
print(f"Break-even precision: {COST_FALSE_POSITIVE/COST_FALSE_NEGATIVE:.2%}")
print("\nAt that ratio you should be willing to talk to a lot of people who were "
      "never going to leave.")

## 2. The uncomfortable truth about this dataset

Before splitting, three things you cannot fix and must therefore state.

In [ ]:
print("A. No time dimension")
print(f"   Columns: {list(df_raw.columns)}")
print("   No date, no snapshot month, no leaving date. `status` is current state.")
print("   -> A temporal split is impossible. You cannot validate the way production works.\n")

print("B. Survivorship in the tenure column")
print(f"   tenure min = {df_raw.tenure.min()}, max = {df_raw.tenure.max()}")
print("   Nobody has under 2 years. Either a filter was applied upstream, or tenure is")
print("   rounded. Every conclusion about early attrition is unavailable from this file.\n")

print("C. The label has no horizon")
print(f"   Base rate: {(df_raw.status=='Left').mean():.1%} have left, at some unknown point")
print("   'Will leave' and 'will leave in the next 6 months' are different problems")
print("   with different models. This file cannot distinguish them.")

**READ and answer.** None of the above is a reason to stop. It is a reason to be
precise about what you will claim at the end.

> *Write the one sentence you would put in your findings to caveat all three.*

A practitioner states limitations up front, in their own words, before anyone else
finds them. That is what separates a caveat from an excuse.

### So how do we split?

With no time dimension and one row per employee, a **stratified random split** is
the defensible choice. Stratified because the classes are imbalanced, and random
because there is no better ordering available.

What you lose is the ability to detect drift, and that loss goes in the write-up.

In [ ]:
from sklearn.model_selection import train_test_split

train_raw, test_raw = train_test_split(
    df_raw, test_size=0.25, random_state=42, stratify=df_raw.status)

print(f"TRAIN {train_raw.shape}  attrition {(train_raw.status=='Left').mean():.1%}")
print(f"TEST  {test_raw.shape}  attrition {(test_raw.status=='Left').mean():.1%}   <-- SEALED")

df = train_raw.copy()      # everything in section 3 uses training data only

---

# 3. The EDA ritual

Nine steps, same order every time. Steps 1 to 4: what you have. Steps 5 to 8: what
is wrong with it. Step 9: what to do next.

### Step 1 · Shape and duplicates

In [ ]:
print("Rows, columns:", df.shape)
print("Fully duplicated rows:", df.duplicated().sum())

Duplicates matter here in a way they did not in the telco panel. There is no ID
column, so two identical rows are either two genuinely similar employees or the same
person recorded twice. You cannot tell from the file. **Ask.**

### Step 2 · Types and completeness

In [ ]:
df.info()

### Step 3 · Look at the actual rows

In [ ]:
df.head(8)

### Step 4 · Summary statistics

In [ ]:
df.describe().T

In [ ]:
df.select_dtypes(exclude="number").describe().T

**READ and answer.** Look at the `describe()` row for `filed_complaint`.

> *Its count, mean, std, min and max are all telling you the same thing. What is it,
> and is that column actually missing data?*

Section 4 answers this, and it is the single most important idea in the notebook.

### Step 5 · Missing values

In [ ]:
miss = pd.DataFrame({
    "missing": df.isnull().sum(),
    "share": df.isnull().mean(),
}).query("missing > 0").sort_values("missing", ascending=False)
print(miss.assign(share=lambda d: d.share.map("{:.1%}".format)).to_string())

In [ ]:
# Do columns go missing TOGETHER? Overlap as a share of each column's own gaps.
# A pair near 100% both ways shares one upstream cause.
gaps = df.isnull()
cols = list(gaps.columns[gaps.any()])
print("Column pairs that go missing together (>=80% mutual overlap):\n")
found = False
for i, a in enumerate(cols):
    for b in cols[i+1:]:
        both = (gaps[a] & gaps[b]).sum()
        if both == 0: continue
        sa, sb = both / gaps[a].sum(), both / gaps[b].sum()
        if min(sa, sb) >= 0.80:
            found = True
            print(f"  {a} + {b}: {both:,} rows  ({sa:.0%} of {a}'s gaps, {sb:.0%} of {b}'s)")
if not found:
    print("  none - the gaps have independent causes")

**READ and answer.** Two pairs should show up, and they mean completely different
things.

**`satisfaction` + `tenure`** go missing on exactly the same rows. That is a
fingerprint, not a coincidence: both fields come from one source, probably the annual
engagement survey, and for those employees the survey record is absent entirely. One
cause, one fix, one question for HR.

**`filed_complaint` + `recently_promoted`** overlap almost completely, and both are
missing in the high 80s or 90s of a percent.

> *Is the second pair really missing data at all?*

Section 4.1 answers it. The clue is that no sane HR system loses 98% of a field.

### Step 6 · Numeric distributions

In [ ]:
num_cols = ["avg_monthly_hrs", "last_evaluation", "n_projects", "satisfaction", "tenure"]
df[num_cols].hist(figsize=(13, 6), bins=30, color="#0F766E", edgecolor="white")
plt.tight_layout(); plt.show()

### Step 7 · Categorical distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["department", "salary", "status"]):
    sns.countplot(y=col, data=df, order=df[col].value_counts().index, ax=ax, color="#0F766E")
    ax.set_title(col); ax.set_ylabel("")
plt.tight_layout(); plt.show()

print(df.department.value_counts(dropna=False).to_string())

**READ and answer.** Two problems are visible in that department list.

> *Name both before running the next cell.*

In [ ]:
print("PROBLEM 1 - the same department under two labels")
for d in ["IT", "information_technology"]:
    if d in df.department.values:
        print(f"   {d:<24} {(df.department == d).sum():>6,} employees")

print("\nPROBLEM 2 - a department that is not the same kind of thing")
t = df.dropna(subset=["department"]).groupby("department")["avg_monthly_hrs"].mean().sort_values()
print(t.head(3).round(1).to_string())
print(f"\n   Overall mean hours: {df.avg_monthly_hrs.mean():.1f}")

`IT` and `information_technology` are one department recorded by two systems. A
straight relabel.

`temp` is different, and more interesting. Temporary staff work about half the hours
of everyone else, and "attrition" means something else entirely for someone on a
fixed-term contract: they leave because the contract ended, not because they were
unhappy.

Your original notebook flagged this as "irrelevant departments" and left it there.
It is a **population definition** question, and it has to be answered explicitly:

> *Is a temp part of the population this model is meant to serve?*

If HR would never have a retention conversation with a temp, they do not belong in
the training data, and leaving them in teaches the model a pattern it should never
apply. This is a business decision, not a data-cleaning one.

### Step 8 · Segmentation against the target

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, col in zip(axes, num_cols):
    sns.violinplot(x=df[col], y=df.status, ax=ax, hue=df.status,
                   palette={"Employed": "#94A3B8", "Left": "#0F766E"}, legend=False)
    ax.set_title(col); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# The plot that makes this dataset famous. Two features, coloured by outcome.
leavers = df[df.status == "Left"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x="satisfaction", y="last_evaluation", hue="status", data=df,
                palette={"Employed": "#CBD5E1", "Left": "#0F766E"}, s=14, alpha=.6, ax=axes[0])
axes[0].set_title("Everyone")
sns.scatterplot(x="satisfaction", y="last_evaluation", data=leavers,
                color="#0F766E", s=16, alpha=.7, ax=axes[1])
axes[1].set_title(f"Leavers only (n={len(leavers):,})")
plt.tight_layout(); plt.show()

**READ and answer.** Leavers are not one group.

> *How many distinct clusters of leavers can you see, and what is the HR story
> behind each one?*

This matters more than it looks. If there are three routes out of the company, a
single model with a single threshold will serve one of them well and the other two
badly. The honest options are one model with segment-level error analysis, or three
models. Either is defensible. Not noticing is not.

### Step 9 · Correlations, with the usual warning

In [ ]:
# numeric_only=True is REQUIRED on pandas 2.0+. Without it, this raises ValueError
# because `department`, `salary` and `status` are text.
tmp = df.copy()
tmp["left"] = (tmp.status == "Left").astype(int)
corr = tmp[num_cols + ["left"]].corr(numeric_only=True)

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 9}, cbar_kws={"shrink": .8})
plt.title("Correlation, training data only"); plt.tight_layout(); plt.show()

**Correlation measures straight lines.** Look back at the scatter plot: the
relationship between satisfaction and leaving is three clusters, not a line. A low
correlation coefficient here is a limitation of the coefficient, not evidence about
the feature.

Never drop a feature on the strength of this heatmap.

---

# 4. Cleaning

The rule that governs every cell below:

> **Any number you learn from the data is learned from the training data alone,
> then applied unchanged to the test data.**

### 4.1 The distinction your original notebook missed

`filed_complaint` and `recently_promoted` contain `1.0` or `NaN`, and nothing else.
That `NaN` does not mean "we do not know". It means **no**.

These are not missing values. They are **encoded booleans**, and treating them as
missing data is a real error with two consequences: `describe()` reports a mean of
1.0 which is meaningless, and any imputer you apply will invent variation that does
not exist.

`last_evaluation`, `satisfaction`, `tenure` and `department` are genuinely missing.
Different thing, different treatment.

In [ ]:
BOOLEAN_FLAGS = ["filed_complaint", "recently_promoted"]

for c in BOOLEAN_FLAGS:
    print(f"{c:<20} distinct non-null values: {sorted(df[c].dropna().unique())}")
print("\nOnly one value each. NaN is the other half of a binary, not an absence.")

In [ ]:
def decode_flags(d):
    """NaN means no. This is a decode, not an imputation."""
    d = d.copy()
    for c in BOOLEAN_FLAGS:
        d[c] = d[c].fillna(0).astype(int)
    return d

train, test = decode_flags(train_raw), decode_flags(test_raw)
print(train[BOOLEAN_FLAGS].mean().round(4).to_string())
print("\nThose are now real rates: the share who complained, and the share promoted.")

### 4.2 Structural fixes

In [ ]:
DEPARTMENT_MAP = {"information_technology": "IT"}
EXCLUDE_DEPARTMENTS = ["temp"]          # <-- a business decision, recorded in code

def fix_departments(d, exclude=True):
    d = d.copy()
    d["department"] = d["department"].replace(DEPARTMENT_MAP)
    if exclude:
        d = d[~d["department"].isin(EXCLUDE_DEPARTMENTS)].copy()
    return d

before = len(train)
train, test = fix_departments(train), fix_departments(test)
print(f"IT + information_technology merged.")
print(f"Temporary staff excluded: {before - len(train):,} training rows removed.")
print(f"\ntrain {train.shape}  test {test.shape}")

Two things to notice about that cell.

**The mapping is a named constant.** Not a chain of `.replace()` calls buried in the
middle of the notebook. Six months from now somebody needs to know what was merged
and why, and the constant is the record.

**The exclusion is a flag, not a hard-coded filter.** Set `exclude=False` and you can
test how much the decision mattered. A judgement call you cannot reverse is a
judgement call you cannot defend.

### 4.3 Genuine missing values: indicator first, then fill

In [ ]:
# READ: is missingness informative? Check before deciding how to treat it.
tr_left = (train.status == "Left")
print(f"{'column':<17}{'missing':>9}{'attrition present':>19}{'attrition missing':>19}{'verdict':>22}")
print("-" * 86)
for c in ["last_evaluation", "satisfaction", "tenure", "department"]:
    m = train[c].isnull()
    if m.sum() == 0: continue
    a, b = tr_left[~m].mean(), tr_left[m].mean()
    ratio = b / a if a else 1
    verdict = "keep the indicator" if abs(ratio - 1) > 0.20 else "indicator adds little"
    print(f"{c:<17}{m.mean():>8.1%}{a:>19.1%}{b:>19.1%}{verdict:>22}")

**READ and answer.** Compare the last two columns for each row.

> *For which of these is the fact of being missing worth keeping as a feature?*

Where the two rates differ materially, missingness carries signal and the indicator
earns its place. Where they match, the indicator is close to noise.

On this data the verdicts are likely to be unflattering, and that is a real result:
**not every indicator is worth keeping, and you only know by checking.** The habit is
to check, not to create them reflexively.

Either way: **create the indicator before you fill.** Once you fill, the information
is gone and the check becomes impossible.

In [ ]:
FILL = {}          # learned from TRAIN only

def handle_missing(d, fit=False):
    d = d.copy()
    for c in ["last_evaluation", "satisfaction", "tenure"]:
        d[f"{c}_missing"] = d[c].isnull().astype(int)        # indicator FIRST
        if fit:
            FILL[c] = d.loc[d[c].notnull(), c].median()
        d[c] = d[c].fillna(FILL[c])                          # then fill
    d["department"] = d["department"].fillna("Unknown")
    return d

train = handle_missing(train, fit=True)      # learn the medians here
test  = handle_missing(test,  fit=False)     # apply them unchanged here

print("Medians learned from training data:", {k: round(v, 3) for k, v in FILL.items()})
print("Remaining nulls -> train:", train.isnull().sum().sum(), "| test:", test.isnull().sum().sum())

**This is the cell that matters.**

`fit=True` computes the medians on train. `fit=False` applies them to test. Compute
them across both and the test set has influenced a number used to build the model,
and your test score stops being an honest estimate of anything.

Note also the pandas 3.0-safe assignment. Your original used
`df.satisfaction.fillna(0, inplace=True)`, which is chained assignment on a column:
deprecated in pandas 2.2 and broken under copy-on-write. It also filled with **zero**,
which places those employees at the bottom of a 0-to-1 satisfaction scale rather than
in the middle. That is not a neutral choice, it is a strong and wrong claim.

---

# 5. Enrichment

Your original notebook created `unhappy`, `underperformer` and `overachiever` with
hard-coded cutoffs, computed on the full dataframe.

We keep the ideas, which are good, and fix three things: the thresholds are learned
from training data, they are applied through one function to both sets, and each one
is checked for whether it actually separates the outcome.

In [ ]:
CUT = {}

def enrich(d, fit=False):
    d = d.copy()
    if fit:
        CUT["low_satisfaction"]  = d.satisfaction.quantile(0.20)
        CUT["weak_evaluation"]   = d.last_evaluation.quantile(0.25)
        CUT["strong_evaluation"] = d.last_evaluation.quantile(0.80)
        CUT["long_hours"]        = d.avg_monthly_hrs.quantile(0.80)
        CUT["short_hours"]       = d.avg_monthly_hrs.quantile(0.20)

    d["unhappy"] = (d.satisfaction < CUT["low_satisfaction"]).astype(int)
    # the original counted a filled-in blank as underperformance; exclude those
    d["underperformer"] = ((d.last_evaluation < CUT["weak_evaluation"]) &
                           (d.last_evaluation_missing == 0)).astype(int)
    d["overachiever"] = ((d.last_evaluation > CUT["strong_evaluation"]) &
                         (d.satisfaction > CUT["low_satisfaction"])).astype(int)
    d["overworked"]  = (d.avg_monthly_hrs > CUT["long_hours"]).astype(int)
    d["underused"]   = (d.avg_monthly_hrs < CUT["short_hours"]).astype(int)

    # combinations, which is where the real signal lives
    d["burnout_risk"]  = (d.overworked & d.unhappy).astype(int)
    d["flight_risk"]   = (d.overachiever & d.overworked).astype(int)
    d["stagnant"]      = ((d.tenure >= 4) & (d.recently_promoted == 0)).astype(int)
    d["hours_per_project"] = d.avg_monthly_hrs / d.n_projects.clip(lower=1)
    return d

train = enrich(train, fit=True)
test  = enrich(test,  fit=False)
print("Thresholds learned from training data:")
for k, v in CUT.items():
    print(f"  {k:<20} {v:.3f}")

### Does each engineered feature actually do anything?

In [ ]:
base = (train.status == "Left").mean()
rows = []
for f in ["unhappy","underperformer","overachiever","overworked","underused",
          "burnout_risk","flight_risk","stagnant","filed_complaint","recently_promoted"]:
    flagged = train[train[f] == 1]
    if len(flagged) == 0: continue
    rows.append({"feature": f, "share flagged": (train[f] == 1).mean(),
                 "attrition if flagged": (flagged.status == "Left").mean(),
                 "lift vs base": (flagged.status == "Left").mean() / base})

lift = pd.DataFrame(rows).sort_values("lift vs base", ascending=False)
print(f"Base attrition rate: {base:.1%}\n")
print(lift.round(3).to_string(index=False))

**READ and answer.** A lift near 1.0 means the feature separates nothing.

> *Which engineered features would you drop, and which surprised you?*

Feature engineering is not measured by how many columns you produced. A feature that
does not separate the outcome is not neutral: it adds variance, it slows training,
and it gives somebody a reason to build a story around a coefficient that means
nothing.

### FILL: the cost of learning thresholds from everything

In [ ]:
# FILL: the same threshold, learned the way most people do it.
pooled = pd.concat([train, test], ignore_index=True)
leaky  = pooled.satisfaction.quantile(0.20)
honest = CUT["low_satisfaction"]

flips = (test.satisfaction < honest) != (test.satisfaction < leaky)
print(f"Threshold from training data only : {honest:.4f}")
print(f"Threshold from train + test pooled: {leaky:.4f}")
print(f"Test rows whose `unhappy` flag flips: {flips.sum():,} of {len(test):,}")

**READ and answer.** The difference is probably small.

> *So why is it still wrong?*

Because the size of the effect is not knowable in advance. "It was only a few rows"
is a conclusion you can reach only after you have already compromised the test set.
And the habit does not stay small: the same shortcut applied to a target-encoded
category, or a scaler fitted on everything, produces test scores that are badly and
invisibly inflated.

Correctness here is free. Discovering the problem late is not.

## 6. Save the analytical base table

In [ ]:
TARGET = "left"
for d in (train, test):
    d[TARGET] = (d.status == "Left").astype(int)

drop = ["status"]
train_abt = train.drop(columns=drop)
test_abt  = test.drop(columns=drop)

train_abt.to_csv("employee_abt_train.csv", index=False)
test_abt.to_csv("employee_abt_test.csv",  index=False)

print(f"employee_abt_train.csv  {train_abt.shape}")
print(f"employee_abt_test.csv   {test_abt.shape}")
print(f"\nAttrition -> train {train_abt[TARGET].mean():.1%} | test {test_abt[TARGET].mean():.1%}")
print(f"\nColumns: {list(train_abt.columns)}")

---

## What to carry into modelling

**Four things you must state in any write-up from this data.**

1. There is no time dimension, so the validation is a random split and drift is
   undetectable from this file.
2. `tenure` starts at 2, so early attrition is invisible and any claim about new
   joiners is unsupported.
3. The label has no horizon. This predicts "has left", not "will leave within N
   months", and those are different problems.
4. Temporary staff were excluded as a population decision. Say who agreed it.

**Three things the cleaning taught.**

`NaN` is not always missing. Here it was the false half of two booleans, and
imputing it would have manufactured variation that does not exist.

Missingness has a fingerprint. `satisfaction` and `tenure` vanish together, which
points at one upstream source and one question for HR.

Every learned number is a leak waiting to happen. Medians, quantiles, category
mappings: learn on train, apply to test, through one named function.

**One thing the EDA taught.** Leavers are not one group. Carry that into model
selection and into error analysis, because an aggregate metric will hide it.